In [8]:
import os
import json
import asyncio
import re
import random
import hashlib
import copy
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from dataclasses import dataclass, field  # <--- LA LIGNE MANQUANTE EST LÀ

from pydantic import BaseModel, Field
from mistralai import Mistral
from dotenv import load_dotenv
import time

try:
    from temporalio import activity
except ImportError:
    class activity:
        @staticmethod
        def defn(func): return func
# =============================================================================
# 1. INITIALISATION & CLÉS
# =============================================================================
env_path = Path("cle.env").absolute()
load_dotenv(dotenv_path=env_path, override=True)

if not os.getenv("MISTRAL_API_KEY"): print(f"❌ Erreur : Impossible de trouver les clés dans {env_path}")
else: print(f"✅ Clés chargées depuis {env_path}")

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

_FAST_MODEL = "mistral-small-latest"
_SMART_MODEL = "mistral-medium-latest"
_BEST_MODEL = "mistral-large-latest"

# 🧠 TES GARDE-FOUS
CACHE_REQUETES = {}
CACHE_RESULTATS_GLOBAUX = {}
SOCIAL_BLACKLIST = ["tiktok.com", "facebook.com", "instagram.com", "x.com", "twitter.com"]
SOURCES_TIER_1 = ["insee.fr", "gouv.fr", "ameli.fr", "vie-publique.fr", "senat.fr", "assemblee-nationale.fr", "ccomptes.fr"]
SOURCES_TIER_2 = ["lemonde.fr", "lefigaro.fr", "liberation.fr", "lesechos.fr", "afp.com", "francetvinfo.fr"]
FRENCH_STOPWORDS = {"alors", "avec", "avoir", "bien", "cette", "dans", "dont", "elle", "elles", "entre", "etre", "fait", "faire", "mais", "meme", "nous", "pour", "plus", "pas", "que", "qui", "sans", "sont", "sur", "tout", "tous", "tres", "une", "des", "les", "du", "de", "la", "le", "un", "est", "et", "ou", "en", "il", "ils", "on", "je", "tu", "vous"}

MISTRAL_RATE_LIMIT_MAX_RETRIES = 6
MISTRAL_RATE_LIMIT_BACKOFF_BASE_SECONDS = 0.7
MISTRAL_RATE_LIMIT_BACKOFF_MAX_SECONDS = 20.0

# =============================================================================
# 2. TON MOTEUR DE RECHERCHE PERSONNALISÉ (RÉINTÉGRÉ !)
# =============================================================================
def _is_rate_limited_error(exc: Exception) -> bool:
    lowered = str(exc).lower()
    return "rate limit" in lowered or "status 429" in lowered or "1300" in lowered

def _tokenize(text: str) -> list[str]:
    return [t for t in re.findall(r"[a-zA-ZÀ-ÿ0-9]+", (text or "").lower()) if len(t) >= 3 and t not in FRENCH_STOPWORDS]

def _is_valid_http_url(url: str) -> bool:
    return isinstance(url, str) and (url.strip().lower().startswith("http://") or url.strip().lower().startswith("https://"))

def _domain_to_organization(url: str) -> str:
    try: host = urlparse(url).netloc.lower()
    except Exception: host = ""
    return host.replace("www.", "") if host else "source-inconnue"

def _score_source_url(url: str) -> float:
    url_lower = url.lower()
    if any(domain in url_lower for domain in SOURCES_TIER_1): return 1000.0
    if any(domain in url_lower for domain in SOURCES_TIER_2): return 500.0
    return 10.0

def generer_requete_deterministe(affirmation: str, prefix: str) -> str:
    phrase_id = hashlib.md5(affirmation.strip().lower().encode()).hexdigest()
    cache_key = f"{prefix}_{phrase_id}"
    if cache_key in CACHE_REQUETES:
        return CACHE_REQUETES[cache_key]
    mots_cles = " ".join(_tokenize(affirmation))
    requete_stricte = f"{prefix} {mots_cles} France"
    CACHE_REQUETES[cache_key] = requete_stricte
    return requete_stricte

async def _mistral_web_search_response(query: str) -> Any:
    print(f"   🔎 Recherche Web Mistral en cours : '{query[:40]}...'")
    for attempt in range(MISTRAL_RATE_LIMIT_MAX_RETRIES + 1):
        try:
            return await client.beta.conversations.start_async(
                model="mistral-medium-latest", inputs=query, tools=[{"type": "web_search"}],
                completion_args={"temperature": 0.0, "random_seed": 42}
            )
        except Exception as exc:
            if _is_rate_limited_error(exc) and attempt < MISTRAL_RATE_LIMIT_MAX_RETRIES:
                pause = min(MISTRAL_RATE_LIMIT_BACKOFF_MAX_SECONDS, MISTRAL_RATE_LIMIT_BACKOFF_BASE_SECONDS * (2 ** attempt))
                print(f"   ⏳ Rate limit Web Search. Pause {pause:.1f}s...")
                await asyncio.sleep(pause)
            else: raise

def _extract_mistral_web_candidates(response: Any) -> list[dict[str, str]]:
    payload = response.model_dump() if hasattr(response, "model_dump") else response
    candidates = []
    for output in payload.get("outputs", []):
        if output.get("type") == "message.output" and isinstance(output.get("content"), list):
            for chunk in output["content"]:
                if isinstance(chunk, dict) and chunk.get("type") == "tool_reference" and _is_valid_http_url(chunk.get("url", "")):
                    candidates.append({"url": chunk["url"], "title": chunk.get("title", ""), "snippet": chunk.get("title", "")[:480]})
    return candidates

async def _search_relevant_sources(assertion: str, question: str, query: str, max_results: int = 6, allow_social: bool = False) -> list[dict[str, str]]:
    try:
        response = await _mistral_web_search_response(f"{query}\nTrouve des pages pertinentes.")
        raw_candidates = _extract_mistral_web_candidates(response)
    except Exception as exc:
        print(f"   ❌ Erreur Web Search: {exc}")
        return []

    filtered = []
    for item in raw_candidates:
        host = _domain_to_organization(item["url"])
        if not allow_social and any(host == b or host.endswith(f".{b}") for b in SOCIAL_BLACKLIST): continue
        filtered.append(item)
        if len(filtered) >= max_results: break

    # Scoring
    needed_tokens = set(_tokenize(f"{assertion} {question}"))
    scored = []
    for idx, c in enumerate(filtered):
        haystack = " ".join([c["title"], c["url"]])
        overlap = len(needed_tokens.intersection(set(_tokenize(haystack))))
        score = (overlap / max(1, len(needed_tokens))) * _score_source_url(c["url"])
        scored.append((score, c))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{"organization": _domain_to_organization(s["url"]), "url": s["url"], "snippet": s["snippet"]} for _, s in scored[:3]]

def _sources_for_prompt(sources: list[dict]) -> str:
    if not sources: return "Aucune source trouvée."
    return "\n".join([f"[{i+1}] org={s['organization']} url={s['url']} snippet={s.get('snippet', '')}" for i, s in enumerate(sources)])

def _sanitize_primary_source(raw_result: dict, sources: list[dict]) -> dict:
    sanitized = dict(raw_result)
    if not sources:
        sanitized.update({"source_is_relevant": False, "nom_source": "", "url_source": "", "sources": []})
        return sanitized
    sanitized.update({
        "source_is_relevant": True,
        "nom_source": sources[0]["organization"],
        "url_source": sources[0]["url"],
        "sources": sources
    })
    return sanitized

# =============================================================================
# 3. SCHÉMAS PYDANTIC DU COLLÈGUE (SÉCURISÉS)
# =============================================================================
class RouteurOutput(BaseModel):
    affirmation_propre: str = ""
    run_stats: bool = False
    run_rhetorique: bool = False
    run_coherence_personnelle: bool = False
    run_contexte: bool = False

class SourceEntry(BaseModel):
    url: str = ""
    organization: str = ""

class StatistiqueOutput(BaseModel):
    agent: str = "statistique"
    verdict: str = "indetermine"
    analyse_detaillee: str = ""
    sources: list[SourceEntry] = Field(default_factory=list)

class ContexteOutput(BaseModel):
    agent: str = "contexte"
    analyse_detaillee: str = ""
    sources: list[SourceEntry] = Field(default_factory=list)

class CoherenceOutput(BaseModel):
    agent: str = "coherence"
    explication: str = ""
    sources: list[SourceEntry] = Field(default_factory=list)

class RhetoriqueOutput(BaseModel):
    agent: str = "rhetorique"
    explication: str = ""

class ExplanationWithSource(BaseModel):
    texte: str = ""
    source: str = ""
    url: str = ""

class EditorExplications(BaseModel):
    statistique: ExplanationWithSource | None = None
    contexte: ExplanationWithSource | None = None
    rhetorique: ExplanationWithSource | None = None
    coherence: ExplanationWithSource | None = None

class EditorOutput(BaseModel):
    verdict_global: str = ""
    explications: EditorExplications | None = None
    raison: str = ""

def response_format_for_model(schema_name: str, model_cls: type[BaseModel]) -> dict:
    return {"type": "json_schema", "json_schema": {"name": schema_name, "schema": model_cls.model_json_schema(), "strict": True}}

# =============================================================================
# 4. TES PROMPTS DICTATORIAUX & AGENTS EXPERTS
# =============================================================================
def build_routeur_prompt(data: dict) -> str:
    return f"""Tu es un automate de classification. Question: '{data.get('question_posee','')}' | Phrase: '{data['affirmation']}'
    ARBRE DE DÉCISION (UN SEUL TRUE) :
    1. STATISTIQUE: chiffre, %, ou quantité (aucun, zéro, tous, millions). -> run_stats: true
    2. CONTEXTE: événement, loi, lieu. -> run_contexte: true
    3. RHÉTORIQUE: esquive d'une question. -> run_rhetorique: true
    4. COHÉRENCE: passé de la personne. -> run_coherence_personnelle: true
    """

def build_stat_prompt(data: dict, sources_text: str) -> str:
    # 'data' contiendra ici l'affirmation_propre (nettoyée par le routeur)
    return f"""Vérifie : '{data['affirmation']}'
    SOURCES TROUVÉES (OBLIGATOIRE DE LES UTILISER) :
    {sources_text}
    Donne un verdict (vrai, faux, exagéré, trompeur) basé sur les sources.
    """

def build_contexte_prompt(data: dict, sources_text: str) -> str:
    return f"""Donne le contexte de : '{data['affirmation']}'
    SOURCES TROUVÉES :
    {sources_text}
    Explique le vrai contexte en te basant sur les sources.
    """

def build_coherence_prompt(data: dict, sources_text: str) -> str:
    return f"""Vérifie si {data['personne']} se contredit sur : '{data['affirmation']}'
    ARCHIVES TROUVÉES :
    {sources_text}
    Si contradiction, cite l'archive. Sinon, laisse l'explication vide.
    """

def build_editor_prompt(affirmation_actuelle: str, rapports_agents: list) -> str:
    # J'ai renommé les arguments pour qu'ils correspondent EXACTEMENT aux variables de la f-string
     return f"""Tu es le switch logique final d'un logiciel de régie TV.
    
    AFFIRMATION : "{affirmation_actuelle}"
    RAPPORTS : {json.dumps(rapports_agents, ensure_ascii=False)}

    RÈGLE DE FORMATAGE ABSOLUE POUR "texte" (1 seule phrase, AUCUNE EXCEPTION) :
    - CAS STATISTIQUE FAUSSE : Tu DOIS écrire "FAUX : La réalité est de [Le Vrai Chiffre]."
    - CAS STATISTIQUE EXAGÉRÉE : Tu DOIS écrire "EXAGÉRÉ : Le chiffre réel est [Le Vrai Chiffre]."
    - CAS CONTEXTE TROMPEUR : Tu DOIS écrire "TROMPEUR : [Une phrase très courte qui rétablit le fait]."
    - CAS ESQUIVE : Tu DOIS écrire "ESQUIVE : L'invité ne répond pas sur [Le sujet de la question]."
    - CAS VRAI / INCERTAIN / MANQUE DE PREUVE : Laisse "afficher_bandeau" à false.

    INTERDICTION : Ne fais aucune phrase d'introduction. Ne mets AUCUNE source dans le champ "texte" (elles iront ailleurs). Sois concis et brut.
    
    FORMAT JSON STRICT ATTENDU :
    {{
      "afficher_bandeau": true,
      "verdict_global": "Faux",
      "explications": {{
         "statistique": {{
            "texte": "FAUX : La réalité est de 5,5 %."
         }},
         "contexte": {{
            "texte": "TROMPEUR : La loi a été rejetée le mois dernier."
         }}
      }}
    }}
    NOTE : N'inclus dans 'explications' que l'agent pertinent. Laisse les champs URL/Source vides dans le prompt (le code Python s'en charge).
    """
# =============================================================================
# 5. POOL D'AGENTS (SANS L'OUTIL WEB_SEARCH QUI PLANTAIT)
# =============================================================================
@dataclass
class AgentPool:
    specialist_ids: dict[str, str]

# ⚠️ On a retiré `"tools": [{"type": "web_search"}]` ! L'agent lit juste le prompt et répond en 1 seconde.
AGENT_DEFINITIONS = {
    "routeur": {"model": _FAST_MODEL, "schema_name": "routeur_output", "model_cls": RouteurOutput},
    "statistique": {"model": _SMART_MODEL, "schema_name": "statistique_output", "model_cls": StatistiqueOutput},
    "contexte": {"model": _SMART_MODEL, "schema_name": "contexte_output", "model_cls": ContexteOutput},
    "coherence": {"model": _SMART_MODEL, "schema_name": "coherence_output", "model_cls": CoherenceOutput},
    "rhetorique": {"model": _FAST_MODEL, "schema_name": "rhetorique_output", "model_cls": RhetoriqueOutput},
    "editeur": {"model": _BEST_MODEL, "schema_name": "editor_output", "model_cls": EditorOutput},
}

_POOL_INSTANCE: AgentPool | None = None

async def init_agent_pool() -> AgentPool:
    print("⏳ Création du Pool d'Agents Mistral...")
    specialist_ids = {}
    for key, defi in AGENT_DEFINITIONS.items():
        res = await client.beta.agents.create_async(
            name=f"tv-agent-{key}-{os.urandom(2).hex()}",
            model=defi["model"],
            completion_args={"temperature": 0.0, "random_seed": 42, "response_format": response_format_for_model(defi["schema_name"], defi["model_cls"])}
        )
        specialist_ids[key] = res.id
        print(f" ✅ Agent {key} prêt ({res.id[:8]}...)")
    global _POOL_INSTANCE
    _POOL_INSTANCE = AgentPool(specialist_ids=specialist_ids)
    return _POOL_INSTANCE

async def run_task(agent_key: str, prompt: str) -> dict:
    agent_id = _POOL_INSTANCE.specialist_ids[agent_key]
    print(f"   🧠 [Agent {agent_key}] Analyse en cours (1 tour)...")
    for attempt in range(5):
        try:
            res = await client.beta.conversations.start_async(agent_id=agent_id, inputs=prompt)
            for o in reversed(res.model_dump().get("outputs", [])):
                if o.get("type") == "message.output" and o.get("content"):
                    texte = str(o["content"]).strip().replace("```json", "").replace("```", "")
                    try: return json.loads(texte)
                    except:
                        m = re.search(r"\{.*\}", texte, flags=re.DOTALL)
                        return json.loads(m.group(0)) if m else {}
            return {}
        except Exception as e:
            if "429" in str(e) and attempt < 4: await asyncio.sleep(2**(attempt+1))
            else: return {}

# =============================================================================
# 6. EXÉCUTION PARALLÈLE SÉCURISÉE
# =============================================================================
async def agent_statistique(data):
    query = generer_requete_deterministe(data['affirmation'], "statistiques officielles")
    sources = await _search_relevant_sources(data['affirmation'], data.get('question_posee',''), query)
    raw = await run_task("statistique", build_stat_prompt(data, _sources_for_prompt(sources)))
    return _sanitize_primary_source(raw, sources)

async def agent_contexte(data):
    query = generer_requete_deterministe(data['affirmation'], "contexte historique loi")
    sources = await _search_relevant_sources(data['affirmation'], data.get('question_posee',''), query)
    raw = await run_task("contexte", build_contexte_prompt(data, _sources_for_prompt(sources)))
    return _sanitize_primary_source(raw, sources)

async def agent_coherence(data):
    query = generer_requete_deterministe(f"citation {data['personne']} {data['affirmation']}", "archives")
    sources = await _search_relevant_sources(data['affirmation'], data.get('question_posee',''), query, allow_social=True)
    raw = await run_task("coherence", build_coherence_prompt(data, _sources_for_prompt(sources)))
    return _sanitize_primary_source(raw, sources)

@activity.defn
async def analyze_debate_line(data: dict) -> dict:
    phrase_id = hashlib.md5(data['affirmation'].strip().lower().encode()).hexdigest()
    if phrase_id in CACHE_RESULTATS_GLOBAUX:
        print(f"⚡ [CACHE HIT GLOBAL] Résultat rechargé !")
        return CACHE_RESULTATS_GLOBAUX[phrase_id]

    routage = await run_task("routeur", build_routeur_prompt(data))
    data["affirmation"] = routage.get("affirmation_propre", data["affirmation"])
    print(f"✨ NETTOYÉ : '{data['affirmation']}'")

    taches = []
    if routage.get("run_stats"): taches.append(agent_statistique(data))
    elif routage.get("run_contexte"): taches.append(agent_contexte(data))
    elif routage.get("run_coherence_personnelle"): taches.append(agent_coherence(data))
    elif routage.get("run_rhetorique"): taches.append(run_task("rhetorique", build_rhetorique_prompt(data)))

    rapports_bruts = await asyncio.gather(*taches) if taches else []
    print(f"   📊 RAPPORTS BRUTS : {[r.get('agent') for r in rapports_bruts]}")

    resultat_final = await run_task("editeur", build_editor_prompt(data["affirmation"], rapports_bruts))
    
    # Validation du bandeau et sources
    afficher = False
    for k, v in resultat_final.get("explications", {}).items():
        if isinstance(v, dict) and v.get("texte"): 
            afficher = True
            rapport = next((r for r in rapports_bruts if r.get("agent") == k), None)
            if rapport and rapport.get("sources"):
                v["url"] = rapport["sources"][0].get("url", "")
                v["source"] = rapport["sources"][0].get("organization", "Source")

    resultat_final["afficher_bandeau"] = afficher
    CACHE_RESULTATS_GLOBAUX[phrase_id] = resultat_final
    return resultat_final

# =============================================================================
# 7. BENCHMARK DE TEST
# =============================================================================
dataset_pipeline_expert = [
    {"p": "Gabriel Attal", "q": "Que faites-vous pour les sans-abris ?", "a": "Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui."},
    {"p": "Jordan Bardella", "q": "Quelle est votre position sur l'énergie ?", "a": "Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%."}
]

async def run_benchmark_tv_production():
    if not _POOL_INSTANCE: await init_agent_pool()
    print("="*80)

    for i, item in enumerate(dataset_pipeline_expert):
        try:
            start_time = time.perf_counter()
            print(f"\n🎬 TEST {i+1}/{len(dataset_pipeline_expert)} | Sujet : {item['p']}")
            print(f"🗣️  PHRASE : '{item['a']}'")

            bandeau_tv_final = await analyze_debate_line({"personne": item['p'], "question_posee": item.get('q', ''), "affirmation": item['a']})

            if bandeau_tv_final.get('afficher_bandeau'):
                explications = bandeau_tv_final.get('explications', {})
                for k, exp in explications.items():
                    if isinstance(exp, dict) and exp.get("texte"):
                        print(f"🚨 [ENVOI OBS] -> {exp['texte']} (Source : {exp.get('source', 'Inconnue')})")
                        break
            else:
                print(f"🔇 [SILENCE OBS] Pas de fait pertinent ou verdict VRAI.")

            print(f"⏱️  Temps : {time.perf_counter() - start_time:.2f}s")
        except Exception as e:
            print(f"❌ Erreur critique au test {i+1} : {e}")

        print("-" * 80)
        await asyncio.sleep(8) 

await run_benchmark_tv_production()

✅ Clés chargées depuis C:\Users\emmag\hackathon-paris\cle.env
⏳ Création du Pool d'Agents Mistral...
 ✅ Agent routeur prêt (ag_019ca...)
 ✅ Agent statistique prêt (ag_019ca...)
 ✅ Agent contexte prêt (ag_019ca...)
 ✅ Agent coherence prêt (ag_019ca...)
 ✅ Agent rhetorique prêt (ag_019ca...)
 ✅ Agent editeur prêt (ag_019ca...)

🎬 TEST 1/2 | Sujet : Gabriel Attal
🗣️  PHRASE : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
   🧠 [Agent routeur] Analyse en cours (1 tour)...
✨ NETTOYÉ : ''
   🔎 Recherche Web Mistral en cours : 'statistiques officielles  France
Trouve ...'
   🧠 [Agent statistique] Analyse en cours (1 tour)...
   📊 RAPPORTS BRUTS : ['statistique']
   🧠 [Agent editeur] Analyse en cours (1 tour)...
🔇 [SILENCE OBS] Pas de fait pertinent ou verdict VRAI.
⏱️  Temps : 35.03s
--------------------------------------------------------------------------------

🎬 TEST 2/2 | Sujet : Jordan Bardella
🗣️  PHRASE : 'Aujourd'hui, le nucléaire ne représente plus que 20% de

In [11]:
import os
import json
import asyncio
import re
import hashlib
import copy
import time
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from mistralai import Mistral
from dotenv import load_dotenv

# --- SÉCURITÉ IMPORTS ---
try:
    from temporalio import activity
except ImportError:
    class activity:
        @staticmethod
        def defn(func): return func

# =============================================================================
# 1. SETUP & GARDE-FOUS
# =============================================================================
env_path = Path("cle.env").absolute()
load_dotenv(dotenv_path=env_path, override=True)
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

_FAST_MODEL = "mistral-small-latest"
_SMART_MODEL = "mistral-medium-latest"
_BEST_MODEL = "mistral-large-latest"

CACHE_RESULTATS_GLOBAUX = {}
SOCIAL_BLACKLIST = ["tiktok.com", "facebook.com", "instagram.com", "x.com", "twitter.com"]
FRENCH_STOPWORDS = {"alors", "avec", "avoir", "bien", "cette", "dans", "dont", "elle", "elles", "entre", "etre", "fait", "faire", "mais", "meme", "nous", "pour", "plus", "pas", "que", "qui", "sans", "sont", "sur", "tout", "tous", "tres", "une", "des", "les", "du", "de", "la", "le", "un", "est", "et", "ou", "en", "il", "ils", "on", "je", "tu", "vous"}

# =============================================================================
# 2. SCHÉMAS PYDANTIC (ARCHITECTURE COLLÈGUE)
# =============================================================================
class RouteurOutput(BaseModel):
    affirmation_propre: str = ""
    run_stats: bool = False
    run_rhetorique: bool = False
    run_coherence_personnelle: bool = False
    run_contexte: bool = False

class SourceEntry(BaseModel):
    url: str = ""
    organization: str = ""

class StatistiqueOutput(BaseModel):
    agent: str = "statistique"
    verdict: str = "indetermine"
    analyse_detaillee: str = ""
    sources: list[SourceEntry] = Field(default_factory=list)

class ContexteOutput(BaseModel):
    agent: str = "contexte"
    analyse_detaillee: str = ""
    sources: list[SourceEntry] = Field(default_factory=list)

class CoherenceOutput(BaseModel):
    agent: str = "coherence"
    explication: str = ""
    sources: list[SourceEntry] = Field(default_factory=list)

class RhetoriqueOutput(BaseModel):
    agent: str = "rhetorique"
    explication: str = ""

class ExplanationWithSource(BaseModel):
    texte: str = ""
    source: str = ""
    url: str = ""

class EditorExplications(BaseModel):
    statistique: ExplanationWithSource | None = None
    contexte: ExplanationWithSource | None = None
    rhetorique: ExplanationWithSource | None = None
    coherence: ExplanationWithSource | None = None

class EditorOutput(BaseModel):
    verdict_global: str = ""
    explications: EditorExplications | None = None
    raison: str = ""

# =============================================================================
# 3. MOTEUR DE RECHERCHE & UTILS (VERSION ROBUSTE)
# =============================================================================
def _tokenize(text: str) -> list[str]:
    return [t for t in re.findall(r"[a-zA-ZÀ-ÿ0-9]+", (text or "").lower()) if len(t) >= 3 and t not in FRENCH_STOPWORDS]

def _domain_to_organization(url: str) -> str:
    try: host = urlparse(url).netloc.lower().replace("www.", "")
    except: host = "source"
    return host

async def _search_relevant_sources(query: str, allow_social: bool = False) -> list[dict]:
    # Sécurité : Si la requête est trop courte, on n'appelle pas Mistral
    if len(query) < 10: return []
    try:
        res = await client.beta.conversations.start_async(
            model=_SMART_MODEL, inputs=query, tools=[{"type": "web_search"}]
        )
        candidates = []
        for o in res.model_dump().get("outputs", []):
            if o.get("type") == "message.output" and isinstance(o.get("content"), list):
                for chunk in o["content"]:
                    if isinstance(chunk, dict) and chunk.get("type") == "tool_reference":
                        url = chunk.get("url", "")
                        host = _domain_to_organization(url)
                        if not allow_social and any(b in host for b in SOCIAL_BLACKLIST): continue
                        candidates.append({"url": url, "organization": host})
        return candidates[:3]
    except: return []

# =============================================================================
# 4. PROMPTS SIMPLIFIÉS (LOGIQUE BINAIRE)
# =============================================================================
def build_stat_prompt(data: dict, sources_text: str) -> str:
    return f"""Vérifie : '{data['affirmation']}'
    SOURCES : {sources_text}
    MISSION : Rédige une analyse de MAXIMUM 15 MOTS. 
    Donne uniquement le chiffre réel. Pas de phrases complexes.
    """

def build_contexte_prompt(data: dict, sources_text: str) -> str:
    return f"""Donne le contexte de : '{data['affirmation']}'
    SOURCES : {sources_text}
    MISSION : Rédige une seule phrase percutante de MAXIMUM 15 MOTS pour rétablir la réalité.
    """

def build_editor_prompt(affirmation: str, rapports: list) -> str:
    return f"""Tu es rédacteur en chef TV. 
    Phrase : '{affirmation}' 
    Rapports experts : {json.dumps(rapports, ensure_ascii=False)}

    RÈGLE ABSOLUE DE FORMATAGE :
    1. Le champ 'texte' doit contenir EXACTEMENT UNE SEULE PHRASE COURTE (max 20 mots).
    2. Format : "[VERDICT] : [Fait brutal]".
    3. INTERDICTION de faire de la pédagogie ou de l'analyse rhétorique (ex: "relève d'une rhétorique trompeuse" est INTERDIT).
    4. Si l'expert est statistique, cite UNIQUEMENT le chiffre.
    5. Si l'expert est contexte, cite UNIQUEMENT le fait contradictoire.

    Exemple Stats : "FAUX : L'INSEE estime à 330 000 le nombre de SDF en France."
    Exemple Contexte : "TROMPEUR : Cette affirmation contredit frontalement les réalités sociales documentées."
    """
# =============================================================================
# 5. POOL D'AGENTS & EXECUTION
# =============================================================================
@dataclass
class AgentPool:
    specialist_ids: dict[str, str]

_POOL_INSTANCE = None

async def init_agent_pool():
    global _POOL_INSTANCE
    print("⏳ Démarrage des agents...")
    ids = {}
    for key, defi in AGENT_DEFINITIONS.items():
        fmt = {"type": "json_schema", "json_schema": {"name": defi["schema"], "schema": defi["cls"].model_json_schema(), "strict": True}}
        res = await client.beta.agents.create_async(name=f"agent-{key}", model=defi["model"], completion_args={"temperature": 0.0, "response_format": fmt})
        ids[key] = res.id
    _POOL_INSTANCE = AgentPool(specialist_ids=ids)

async def run_task(agent_key: str, prompt: str) -> dict:
    try:
        res = await client.beta.conversations.start_async(agent_id=_POOL_INSTANCE.specialist_ids[agent_key], inputs=prompt)
        return json.loads(res.model_dump()["outputs"][-1]["content"])
    except: return {}

AGENT_DEFINITIONS = {
    "routeur": {"model": _FAST_MODEL, "schema": "routeur_output", "cls": RouteurOutput},
    "statistique": {"model": _SMART_MODEL, "schema": "statistique_output", "cls": StatistiqueOutput},
    "contexte": {"model": _SMART_MODEL, "schema": "contexte_output", "cls": ContexteOutput},
    "coherence": {"model": _SMART_MODEL, "schema": "coherence_output", "cls": CoherenceOutput},
    "rhetorique": {"model": _FAST_MODEL, "schema": "rhetorique_output", "cls": RhetoriqueOutput},
    "editeur": {"model": _BEST_MODEL, "schema": "editor_output", "cls": EditorOutput},
}

# =============================================================================
# 6. ANALYSE (LA VERSION QUI NE PLANTE PAS)
# =============================================================================
@activity.defn
async def analyze_debate_line(data: dict) -> dict:
    phrase_id = hashlib.md5(data['affirmation'].strip().lower().encode()).hexdigest()
    if phrase_id in CACHE_RESULTATS_GLOBAUX: return CACHE_RESULTATS_GLOBAUX[phrase_id]

    # 1. ROUTAGE
    routage = await run_task("routeur", build_routeur_prompt(data))
    
    # 🚨 FIX CRITIQUE : Si l'IA a "nettoyé" la phrase en chaîne vide, on force l'originale
    clean_text = routage.get("affirmation_propre")
    if not clean_text or len(clean_text) < 2:
        clean_text = data['affirmation']
    
    # 🛡️ FALLBACK STATS (Si l'IA rate le mot "aucun")
    stats_indicators = ["aucun", "plus aucun", "zéro", "0", "100", "tous", "%"]
    if not any([routage.get("run_stats"), routage.get("run_contexte"), routage.get("run_coherence_personnelle")]):
        if any(word in clean_text.lower() for word in stats_indicators):
            routage["run_stats"] = True

    print(f"✨ TEXTE : '{clean_text}'")

    # 2. EXPERTS
    rapports = []
    if routage.get("run_stats"):
        # On construit une query de recherche solide, même si le texte est court
        query = f"statistiques officielles France {clean_text}"
        srcs = await _search_relevant_sources(query)
        rep = await run_task("statistique", f"Sources: {json.dumps(srcs)}. Affirmation: {clean_text}")
        rep["agent"], rep["sources"] = "statistique", srcs
        rapports.append(rep)
    elif routage.get("run_contexte"):
        srcs = await _search_relevant_sources(f"contexte factuel {clean_text}")
        rep = await run_task("contexte", f"Sources: {json.dumps(srcs)}. Sujet: {clean_text}")
        rep["agent"], rep["sources"] = "contexte", srcs
        rapports.append(rep)
    elif routage.get("run_coherence_personnelle"):
        srcs = await _search_relevant_sources(f"archives déclarations {data['personne']} {clean_text}", allow_social=True)
        rep = await run_task("coherence", f"Sources: {json.dumps(srcs)}. Nom: {data['personne']}")
        rep["agent"], rep["sources"] = "coherence", srcs
        rapports.append(rep)

    # 3. ÉDITION FINALE
    final = await run_task("editeur", build_editor_prompt(clean_text, rapports))
    
    # Mapping sources pour OBS
    afficher = False
    if final.get("explications"):
        for key, exp in final["explications"].items():
            if isinstance(exp, dict) and exp.get("texte"):
                afficher = True
                match = next((r for r in rapports if r.get("agent") == key), None)
                if match and match.get("sources"):
                    exp["url"], exp["source"] = match["sources"][0]["url"], match["sources"][0]["organization"]
    
    final["afficher_bandeau"] = afficher
    CACHE_RESULTATS_GLOBAUX[phrase_id] = final
    return final

async def run_test():
    if not _POOL_INSTANCE: await init_agent_pool()
    dataset = [
        {"personne": "Gabriel Attal", "affirmation": "Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui."},
        {"personne": "Jordan Bardella", "affirmation": "Le nucléaire représente 65% de notre électricité."}
    ]
    
    print(f"\n⏱️  LIMITE PERFORMANCE : 13.00s")
    print("="*60)

    for item in dataset:
        start_time = time.perf_counter() # <--- DÉBUT CHRONO
        print(f"\n🎬 TEST : {item['personne']}")
        
        res = await analyze_debate_line(item)
        
        elapsed = time.perf_counter() - start_time # <--- FIN CHRONO
        
        if res.get("afficher_bandeau"):
            for k, v in res["explications"].items():
                if v and v.get("texte"):
                    print(f"🚨 [OBS] -> {v['texte']} (Source: {v['source']})")
        else:
            print("🔇 [SILENCE]")
            
        color = "\033[92m" if elapsed <= 13 else "\033[91m"
        print(f"⏱️  TEMPS TOTAL : {color}{elapsed:.2f}s\033[0m")
        await asyncio.sleep(2)

await run_test()

⏳ Démarrage des agents...

⏱️  LIMITE PERFORMANCE : 13.00s

🎬 TEST : Gabriel Attal
✨ TEXTE : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
🚨 [OBS] -> FAUX : La Fondation Abbé Pierre estime à 300 000 le nombre de SDF en France. (Source: Fondation Abbé Pierre (2022))
⏱️  TEMPS TOTAL : 12.71s

🎬 TEST : Jordan Bardella
✨ TEXTE : 'Le nucléaire représente 65% de notre électricité.'
🚨 [OBS] -> VRAI : 65% de l'électricité française provient du nucléaire. (Source: optima-energie.fr)
⏱️  TEMPS TOTAL : 23.69s


In [12]:
dataset_pipeline_expert = [
    # --- 5 EXEMPLES AVEC QUESTIONS (Q&A) ---
    {
        "p": "Gabriel Attal", "q": "Que faites-vous pour les sans-abris ?",
        "ctx": "La dignité humaine est au cœur de notre action. Nous avons ouvert 200 000 places d'hébergement d'urgence. L'État mobilise des moyens sans précédent cet hiver. Les préfets ont reçu des instructions claires pour ne laisser personne dehors. Nous avons investi 2 milliards d'euros dans le plan Logement d'Abord. Les associations sont nos partenaires quotidiens. La solidarité nationale doit jouer à plein. Nous ne laissons personne sur le côté. Le combat contre la pauvreté est une priorité.",
        "a": "Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui."
    },
    {
        "p": "Jordan Bardella", "q": "Quelle est votre position sur l'énergie ?",
        "ctx": "La France doit retrouver sa puissance électrique. Le marché européen nous impose des prix délirants qui ruinent les familles. Nous avons un parc nucléaire exceptionnel mais il a été saboté par l'idéologie. Il faut arrêter de fermer des centrales. La souveraineté énergétique est la base de l'indépendance nationale. Nos entreprises délocalisent à cause du coût de l'énergie. Le gaz russe a été remplacé par du gaz de schiste américain plus cher. C'est un non-sens écologique et économique.",
        "a": "Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%."
    },
    {
        "p": "Marine Tondelier", "q": "Faut-il interdire les pesticides ?",
        "ctx": "La santé des Français n'est pas négociable. L'eau que nous buvons est polluée par des molécules chimiques. Les agriculteurs sont les premières victimes de ces produits. On observe une chute dramatique de la biodiversité dans nos campagnes. Les abeilles disparaissent à un rythme alarmant. Le gouvernement recule sans cesse face aux lobbies industriels. Il faut accompagner la transition au lieu de la punir. L'agroécologie est la seule voie de survie pour nos sols.",
        "a": "L'utilisation du glyphosate a augmenté de 50% en France cette année."
    },
    {
        "p": "Bruno Le Maire", "q": "Où en sont les impôts ?",
        "ctx": "Nous avons fait le choix de la politique de l'offre. Baisser les impôts des entreprises permet de créer des emplois. Nous avons supprimé la taxe d'habitation pour tous les Français. La pression fiscale était trop forte dans ce pays. Nous voulons récompenser le travail et l'effort. Les prélèvements obligatoires ont commencé à refluer. C'est une trajectoire de long terme que nous tenons. La croissance revient grâce à cette stabilité fiscale.",
        "a": "Nous avons réussi à diviser par deux la dette de la France en cinq ans."
    },
    {
        "p": "Sandrine Rousseau", "q": "Pourquoi cibler les jets privés ?",
        "ctx": "On demande des efforts de sobriété aux Français modestes. On leur dit de baisser le chauffage à 19 degrés. Mais les ultra-riches continuent de brûler la planète en toute impunité. Un trajet en jet émet plus de CO2 qu'un Français moyen en un an. C'est une injustice climatique insupportable. Le gouvernement refuse de réguler ce secteur. L'écologie ne peut pas être punitive pour les pauvres et permissive pour les riches.",
        "a": "Mais la vraie question, c'est de savoir si le gouvernement va taxer les superprofits de Total !"
    },
    # --- 10 EXEMPLES DISCOURS BRUTS (SANS QUESTIONS) ---
    {
        "p": "Emmanuel Macron", "q": "",
        "ctx": "Nous vivons une période de grandes transformations mondiales. La France doit être aux avant-postes de la technologie. Nous investissons massivement dans l'IA et le quantique. L'école est le berceau de cette ambition future. Il faut réarmer notre éducation nationale. Le niveau en mathématiques doit devenir une priorité dès le CP. Nous avons déjà commencé à recruter davantage de professeurs. La formation continue sera le pilier de notre réforme.",
        "a": "Nous avons recruté 100 000 policiers supplémentaires cette année."
    },
    {
        "p": "Jean-Luc Mélenchon", "q": "",
        "ctx": "Le peuple a faim pendant que les riches se gavent. Les prix à la consommation étranglent les familles. On nous parle de croissance, mais la croissance de quoi ? De la misère ! Nous proposons le blocage des prix des produits de première nécessité. Il faut augmenter le SMIC immédiatement. La vie chère n'est pas une fatalité, c'est un choix politique. Les banques centrales ne font qu'aggraver la situation en montant les taux.",
        "a": "Le SMIC en France est actuellement à 2500 euros net."
    },
    {
        "p": "Marine Le Pen", "q": "",
        "ctx": "La sécurité est la première des libertés. Nos villes et nos villages sont livrés à une délinquance de plus en plus barbare. Le laxisme judiciaire encourage les récidivistes. Il faut rétablir l'autorité partout. Les forces de l'ordre doivent être soutenues par l'État. Nous avons besoin de 85 000 places de prison supplémentaires. Le lien entre immigration et insécurité n'est plus à démontrer pour personne. Les Français veulent vivre en paix.",
        "a": "Le taux de cambriolages a baissé de 40% en un an sous ce gouvernement."
    },
    {
        "p": "Rachida Dati", "q": "",
        "ctx": "La culture est le ciment de notre Nation. Elle ne doit pas être réservée aux élites parisiennes. Chaque village doit avoir accès à une programmation de qualité. Nous soutenons le patrimoine religieux et rural. C'est l'âme de nos territoires. Le pass culture est un outil formidable de démocratisation. Nous allons renforcer les moyens des bibliothèques de quartier. L'art doit entrer dans les écoles dès le plus jeune âge.",
        "a": "Le budget de la culture est le premier poste de dépense de l'État."
    },
    {
        "p": "Olivier Faure", "q": "",
        "ctx": "La réforme des retraites est une blessure sociale qui ne se refermera pas. Travailler jusqu'à 64 ans est une injustice pour ceux qui ont des carrières longues. Les femmes sont les grandes perdantes de cette loi. Nous proposons le retour à la retraite à 60 ans. La pénibilité doit être réellement prise en compte. On ne peut pas traiter un ouvrier comme un cadre. Le financement est possible en taxant les revenus du capital.",
        "a": "La réforme des retraites a été votée à l'unanimité par le Parlement."
    },
    {
        "p": "Eric Ciotti", "q": "",
        "ctx": "La France subit un dérapage budgétaire incontrôlé. Le déficit public atteint des sommets alarmants. Le gouvernement dépense l'argent qu'il n'a pas. Nos enfants hériteront d'une dette colossale. Il faut sabrer dans la dépense publique inutile. L'État doit se concentrer sur ses missions régaliennes. Nous sommes les champions du monde des prélèvements obligatoires. Il faut libérer les entreprises de ce poids mort administratif.",
        "a": "Le déficit public de la France est tombé à 1% du PIB en 2025."
    },
    {
        "p": "Manuel Bompard", "q": "",
        "ctx": "La Ve République est une monarchie présidentielle. Un seul homme décide de tout pour 67 millions de citoyens. L'article 49.3 est devenu l'outil de la brutalité politique. Nous devons passer à une VIe République parlementaire. Le référendum d'initiative citoyenne doit être instauré. Il faut redonner du pouvoir aux élus locaux. La démocratie ne peut pas se résumer à un vote tous les cinq ans. Le peuple doit pouvoir révoquer ses élus.",
        "a": "L'article 49.3 a été utilisé 100 fois cette année."
    },
    {
        "p": "Gérald Darmanin", "q": "",
        "ctx": "La lutte contre les stupéfiants est notre combat quotidien. Les opérations 'place nette' déstabilisent les réseaux. Nous saisissons des quantités records de cocaïne et de cannabis. Les points de deal reculent dans nos quartiers populaires. Nous protégeons les plus jeunes de cette économie souterraine. Les moyens de la gendarmerie sont renforcés. Nous créons 200 nouvelles brigades sur tout le territoire. L'ordre est la condition du progrès social.",
        "a": "Il y a plus de policiers à Paris qu'à New York."
    },
    {
        "p": "Yannick Jadot", "q": "",
        "ctx": "Le climat ne nous attend pas. Les rapports du GIEC sont de plus en plus alarmants. Nous devons sortir des énergies fossiles le plus vite possible. Les transports représentent un tiers de nos émissions. Le train doit devenir moins cher que l'avion. Nous devons isoler tous les bâtiments de ce pays. C'est bon pour la planète et pour le porte-monnaie. L'écologie est une opportunité industrielle majeure pour la France. La nature n'est pas une ressource infinie.",
        "a": "La France est le pays le plus polluant au monde par habitant."
    },
    {
        "p": "François Ruffin", "q": "",
        "ctx": "Le travail doit payer. Aujourd'hui, on peut travailler à temps plein et vivre sous le seuil de pauvreté. C'est indigne. La précarité explose dans les métiers du soin et du lien. Les aides à domicile sont les oubliées de la République. Elles font des kilomètres sans être remboursées. Il faut un salaire minimum européen. Les profits records ne ruissellent pas vers ceux qui produisent. La dignité ouvrière doit être remise au centre.",
        "a": "Le nombre de milliardaires en France a baissé de 50% sous ce mandat."
    }
]

In [13]:
import time
import json
import asyncio

async def run_benchmark_tv_production():
    # 1. On vérifie que le pool d'agents est bien prêt
    if not _POOL_INSTANCE:
        await init_agent_pool()

    print(f"\n⏱️  Lancement du test d'affichage OBS ({len(dataset_pipeline_expert)} tests)...")
    print(f"🎯 LIMITE PERFORMANCE : 13.00s")
    print("="*80)

    for i, item in enumerate(dataset_pipeline_expert):
        try:
            start_time = time.perf_counter()
            
            # Préparation des données (on mappe tes clés 'p', 'q', 'a' vers le format attendu)
            data_input = {
                "personne": item['p'],
                "question_posee": item.get('q', ''),
                "affirmation": item['a'],
                "contexte_precedent": item.get('ctx', '')
            }

            print(f"\n🎬 TEST {i+1}/{len(dataset_pipeline_expert)} | Sujet : {item['p']}")
            print(f"🗣️  PHRASE : '{item['a']}'")
            
            # --- EXÉCUTION DU PIPELINE COMPLET ---
            # Cette fonction gère le Cache, le Routeur, les Experts et l'Éditeur
            bandeau_tv_final = await analyze_debate_line(data_input)

            # --- LOGIQUE D'AFFICHAGE OBS ---
            afficher = bandeau_tv_final.get('afficher_bandeau', False)
            verdict = bandeau_tv_final.get('verdict_global', 'Indéterminé')
            
            if afficher:
                explications = bandeau_tv_final.get('explications', {})
                texte_obs = ""
                source_nom = ""

                # 🏆 SYSTÈME DE PRIORITÉ D'AFFICHAGE (Ordre: Stats -> Contexte -> Cohérence -> Rhétorique)
                if explications.get("statistique") and explications["statistique"].get("texte"):
                    agent_data = explications["statistique"]
                    texte_obs = agent_data["texte"]
                    source_nom = agent_data.get("source", "Source officielle")
                
                elif explications.get("contexte") and explications["contexte"].get("texte"):
                    agent_data = explications["contexte"]
                    texte_obs = agent_data["texte"]
                    source_nom = agent_data.get("source", "Source officielle")
                
                elif explications.get("coherence") and explications["coherence"].get("texte"):
                    agent_data = explications["coherence"]
                    texte_obs = agent_data["texte"]
                    source_nom = agent_data.get("source", "Source officielle")
                
                elif explications.get("rhetorique") and explications["rhetorique"].get("texte"):
                    agent_data = explications["rhetorique"]
                    texte_obs = agent_data["texte"]
                    source_nom = "Analyse Logique"

                if texte_obs:
                    print(f"🚨 [ENVOI OBS] -> {texte_obs} (Source : {source_nom})")
                else:
                    print("🔇 [SILENCE OBS] Le verdict est tombé mais le texte est vide.")
            else:
                raison = bandeau_tv_final.get('raison', 'Pas de contradiction détectée.')
                print(f"🔇 [SILENCE OBS] (Verdict: {verdict}) {raison}")

            # --- CALCUL DU TEMPS ET COULEUR ---
            end_time = time.perf_counter()
            elapsed = end_time - start_time
            
            # Vert si < 13s, Rouge si > 13s
            color = "\033[92m" if elapsed <= 13 else "\033[91m"
            print(f"⏱️  Temps : {color}{elapsed:.2f}s\033[0m")

        except Exception as e:
            print(f"❌ Erreur critique au test {i+1} : {e}")

        # 💤 Pause obligatoire pour éviter le bannissement de l'API (Rate Limit)
        print("-" * 80)
        await asyncio.sleep(10)

# Lancement de la boucle dans le Notebook
await run_benchmark_tv_production()


⏱️  Lancement du test d'affichage OBS (15 tests)...
🎯 LIMITE PERFORMANCE : 13.00s

🎬 TEST 1/15 | Sujet : Gabriel Attal
🗣️  PHRASE : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
🚨 [ENVOI OBS] -> FAUX : La Fondation Abbé Pierre estime à 300 000 le nombre de SDF en France. (Source : Fondation Abbé Pierre (2022))
⏱️  Temps : 0.00s
--------------------------------------------------------------------------------

🎬 TEST 2/15 | Sujet : Jordan Bardella
🗣️  PHRASE : 'Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%.'
✨ TEXTE : 'Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%.'
🚨 [ENVOI OBS] -> FAUX : Le nucléaire représente environ 70% de la production d'électricité en France. (Source : optima-energie.fr)
⏱️  Temps : 15.61s
--------------------------------------------------------------------------------

🎬 TEST 3/15 | Sujet : Marine Tondelier
🗣️  PHRASE : 'L'utilisation du glyphosate a augm